<a href="https://colab.research.google.com/github/yuvipaloozie/CHANA/blob/main/notebooks/Model_Cross_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Evaluation of Models
The goal of this notebook is a comparative analysis between all models beyond simple IoU or accuracy metrics. We are looking to investigate differences in edge cases, cell sizes, counts, etc.

In [ ]:
# CELL 1: SETUP & LIBRARIES
!pip install -q segmentation_models keras-unet-collection scikit-image opencv-python matplotlib seaborn pandas

# UPDATE FOR CELL 1
!pip install -q medpy scipy

from scipy import ndimage
from scipy.stats import wilcoxon
from medpy.metric.binary import hd95
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning) # Suppress medpy warnings for empty masks

import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import numpy as np
import cv2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import DenseNet121, EfficientNetB0
import segmentation_models as sm
from keras_unet_collection import models as unet_models
from skimage import io, measure
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import r2_score, precision_recall_curve, auc
import gc
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

PROJECT_BASE_PATH = '/content/drive/MyDrive/CHANA_files'
WEIGHTS_DIR = os.path.join(PROJECT_BASE_PATH, 'CHANA Final Models and Programs', 'model_weights')
PREPROCESSED_IMAGES_DIR = os.path.join(PROJECT_BASE_PATH, 'preprocessed_rgb')
GROUND_TRUTH_MASKS_DIR = os.path.join(PROJECT_BASE_PATH, 'mask_images_512')

IMG_SIZE = (512, 512)
SEED_VAULT = 999
MIN_AREA = 50
MAX_DISTANCE = 25

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print("✅ Environment Ready.")

In [ ]:
# CELL 2: STRICT DATA RECONSTRUCTION
print("🔓 Securing Test Set Integrity...")

vault_img, vault_mask = [], []

# OPTION A: Load exact Vault used during training (SAFEST)
vault_txt_path = os.path.join(PROJECT_BASE_PATH, 'VAULT_FILES.txt')

if os.path.exists(vault_txt_path):
    print("   -> Found VAULT_FILES.txt. Loading exact test set...")
    with open(vault_txt_path, 'r') as f:
        vault_img = [line.strip() for line in f.readlines()]

    for img_p in vault_img:
        base = os.path.splitext(os.path.basename(img_p))[0]
        for ext in ['_mask.tif', '_mask.png', '.tif', '.png']:
            p = os.path.join(GROUND_TRUTH_MASKS_DIR, base + ext)
            if os.path.exists(p):
                vault_mask.append(p)
                break
else:
    print("   -> VAULT_FILES.txt not found! Re-splitting using SORTED list...")
    all_real_img, all_real_mask = [], []
    # SORTING prevents OS-level randomness
    for f in sorted(os.listdir(PREPROCESSED_IMAGES_DIR)):
        if f.lower().endswith(('.png', '.tif', '.jpg')):
            base = os.path.splitext(f)[0]
            for ext in ['_mask.tif', '_mask.png', '.tif', '.png']:
                p = os.path.join(GROUND_TRUTH_MASKS_DIR, base + ext)
                if os.path.exists(p):
                    all_real_img.append(os.path.join(PREPROCESSED_IMAGES_DIR, f))
                    all_real_mask.append(p)
                    break
    _, vault_img, _, vault_mask = train_test_split(all_real_img, all_real_mask, test_size=0.15, random_state=SEED_VAULT)

print(f"✅ Vault Sealed: {len(vault_img)} images identified.")

def load_test_image(img_path, mask_path):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None: img = io.imread(img_path)
    if img.ndim == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4: img = img[:,:,:3]
    elif img_path.lower().endswith(('.png', '.jpg')): img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img_disp = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img_norm = (img_disp.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None: mask = io.imread(mask_path)
    mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.float32)
    return img_norm, mask, img_disp

In [ ]:
# CELL 3: EXACT ARCHITECTURE BUILDERS & REGISTRY
import tensorflow.keras.applications as applications
print("🏗️ Registering Exact Architectures...")

# --- 1. EXACT TRANSUNET BUILDER (From your training script) ---
BACKBONE_NAME = 'efficientnetb0'
PRETRAINED_WEIGHTS = 'imagenet'
GROUPS = 8

class AddPositionEmbedding(layers.Layer):
    def __init__(self, transformer_dim, max_len=65536, **kwargs):
        super().__init__(**kwargs)
        self.transformer_dim = transformer_dim
        self.pos_embedding = layers.Embedding(input_dim=max_len, output_dim=transformer_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        return x + self.pos_embedding(positions)

def build_transformer_bottleneck(input_tensor, transformer_dim=384, num_heads=6, num_layers=2, output_filters=512):
    x = layers.Conv2D(transformer_dim, (1, 1), padding='same', kernel_initializer='he_normal')(input_tensor)
    x = layers.GroupNormalization(groups=GROUPS)(x)
    patch_seq = layers.Reshape((input_tensor.shape[1] * input_tensor.shape[2], transformer_dim))(x)
    x = AddPositionEmbedding(transformer_dim)(patch_seq)

    for _ in range(num_layers):
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=transformer_dim // num_heads, dropout=0.1)(x, x)
        x = layers.Add()([x, attn_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        ffn_output = layers.Dense(transformer_dim * 4, activation='relu')(x)
        ffn_output = layers.Dense(transformer_dim)(ffn_output)
        x = layers.Add()([x, ffn_output])
        x = layers.LayerNormalization(epsilon=1e-6)(x)

    bottleneck_features = layers.Reshape((input_tensor.shape[1], input_tensor.shape[2], transformer_dim))(x)
    output = layers.Conv2D(output_filters, (1, 1), padding='same', activation='relu', kernel_initializer='he_normal')(bottleneck_features)
    output = layers.GroupNormalization(groups=GROUPS)(output)
    return output

def decoder_block(input_tensor, concat_tensor, num_filters):
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(input_tensor)

    def robust_resize(args):
        img, target = args
        target_shape = tf.shape(target)[1:3]
        return tf.image.resize(img, target_shape)

    x = layers.Lambda(robust_resize)([x, concat_tensor])
    x = layers.concatenate([x, concat_tensor], axis=-1)

    x = layers.Conv2D(num_filters, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.GroupNormalization(groups=GROUPS)(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(num_filters, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.GroupNormalization(groups=GROUPS)(x); x = layers.Activation('relu')(x)
    return x

def build_transunet(input_shape=(512, 512, 3), num_classes=1):
    inputs = Input(shape=input_shape)
    backbone = applications.EfficientNetB0(weights=PRETRAINED_WEIGHTS, include_top=False, input_tensor=inputs)

    backbone.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization): layer.trainable = True

    s1 = backbone.get_layer('block2a_expand_activation').output
    s2 = backbone.get_layer('block3a_expand_activation').output
    s3 = backbone.get_layer('block4a_expand_activation').output
    s4 = backbone.get_layer('block6a_expand_activation').output
    bridge = backbone.get_layer('top_activation').output

    trans_bottleneck = build_transformer_bottleneck(bridge)

    d1 = decoder_block(trans_bottleneck, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    ds1 = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', name='aux1', dtype='float32')(d2)

    d3 = decoder_block(d2, s2, 128)
    ds2 = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', name='aux2', dtype='float32')(d3)

    d4 = decoder_block(d3, s1, 64)
    d5 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(d4)
    d5 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(d5)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', dtype='float32', name='final')(d5)
    return Model(inputs=inputs, outputs=[outputs, ds1, ds2])


# --- 2. EXACT U-NET DENSENET BUILDER (From your script) ---
def build_densenet_unet(input_shape=(512, 512, 3), num_classes=1):
    inputs = Input(shape=input_shape, name='input_layer')
    encoder = DenseNet121(include_top=False, weights=None, input_tensor=inputs)
    def get_layer_output(model, name_options):
        for name in name_options:
            try: return model.get_layer(name).output
            except ValueError: continue
        raise ValueError(f"Could not find layer in {name_options}")
    s1 = get_layer_output(encoder, ['conv1/relu', 'conv1_relu'])
    s2 = get_layer_output(encoder, ['conv2_block6_concat', 'conv2/block6/concat'])
    s3 = get_layer_output(encoder, ['conv3_block12_concat', 'conv3/block12/concat'])
    s4 = get_layer_output(encoder, ['conv4_block24_concat', 'conv4/block24/concat'])
    bridge = get_layer_output(encoder, ['conv5_block16_concat', 'conv5/block16/concat'])

    def dec_block(input_tensor, skip_tensor, filters):
        x = layers.Conv2DTranspose(filters, (2, 2), strides=(2, 2), padding='same')(input_tensor)
        x = layers.Concatenate()([x, skip_tensor])
        x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
        x = layers.Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
        return x

    d1 = dec_block(bridge, s4, 512)
    d2 = dec_block(d1, s3, 256)
    d3 = dec_block(d2, s2, 128)
    d4 = dec_block(d3, s1, 64)
    d5 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(d4)
    d5 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(d5)
    outputs = layers.Conv2D(num_classes, (1, 1), activation='sigmoid', name='final_output', dtype='float32')(d5)
    return Model(inputs=inputs, outputs=outputs)


# --- 3. REGISTRY ---
MODEL_REGISTRY = {
    "TransUNet (Baseline)": {"file": "transunet_domain.weights.h5", "type": "transunet"},
    "TransUNet (Curriculum)":   {"file": "transunet_no_domain.weights.h5", "type": "transunet"},
    "U-Net++ (Baseline)":   {"file": "UNetPlusPlus_Domain.weights.h5", "type": "unetplusplus"},
    "U-Net++ (Curriculum)":     {"file": "Unetplusplus_no_Domain.weights.h5", "type": "unetplusplus"},
    "U-Net (Baseline)":     {"file": "Unet_DenseNet_Domain.weights.h5", "type": "unet"},
    "U-Net (Curriculum)":       {"file": "Unet_DenseNet_no_Domain.weights.h5", "type": "unet"}
}

def load_contender(name):
    cfg = MODEL_REGISTRY[name]
    path = os.path.join(WEIGHTS_DIR, cfg['file'])
    if not os.path.exists(path):
        print(f"  ❌ Missing file: {cfg['file']} in {WEIGHTS_DIR}")
        return None

    print(f"Loading {name}...")
    if cfg['type'] == 'transunet':
        m = build_transunet()
    elif cfg['type'] == 'unetplusplus':
        m = unet_models.unet_plus_2d((512,512,3), [64,128,256,512], 1, stack_num_down=4, stack_num_up=4, activation='ReLU', output_activation='Sigmoid', batch_norm=True, pool=False, unpool=False, backbone='ResNet50', weights=None, deep_supervision=True)
    elif cfg['type'] == 'unet':
        m = build_densenet_unet()

    m.load_weights(path)
    print("  ✅ Weights loaded successfully.")
    return m

In [ ]:
# CELL 4: MASTER INFERENCE LOOP (WITH WATERSHED SEPARATION)
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

print("⚔️ ENTERING THE SOTA ARENA (Watershed Enabled)...")

results = []
MAX_HD_PENALTY = 500.0

def separate_touching_cells(binary_mask, min_dist=20):
    """Uses Watershed to cut connected blobs into distinct instances."""
    distance = ndimage.distance_transform_edt(binary_mask)
    # Find centers of cells (peaks in the distance map)
    local_maxi = peak_local_max(distance, min_distance=min_dist, labels=binary_mask)
    if len(local_maxi) == 0: return measure.label(binary_mask) # Fallback

    # Create markers at the peaks
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(local_maxi.T)] = True
    markers, _ = ndimage.label(mask)

    # Run watershed
    labels = watershed(-distance, markers, mask=binary_mask)
    return labels

for model_name in MODEL_REGISTRY.keys():
    model = load_contender(model_name)
    if model is None: continue

    total_tp, total_fp, total_fn = 0, 0, 0

    for i, (img_p, mask_p) in enumerate(zip(vault_img, vault_mask)):
        img_norm, true_mask, _ = load_test_image(img_p, mask_p)
        if img_norm is None: continue

        preds = model.predict(np.expand_dims(img_norm, 0), verbose=0)

        if MODEL_REGISTRY[model_name]['type'] == 'transunet': pred_prob = preds[0][0,:,:,0]
        elif MODEL_REGISTRY[model_name]['type'] == 'unetplusplus': pred_prob = preds[-1][0,:,:,0]
        else: pred_prob = preds[0,:,:,0]

        pred_prob = np.squeeze(pred_prob)
        t_mask = np.squeeze(true_mask).astype(np.uint8)

        # Threshold & Fill Holes
        p_mask = (pred_prob > 0.5).astype(np.uint8)
        p_mask = ndimage.binary_fill_holes(p_mask).astype(np.uint8)

        # 1. Pixel IoU & HD95
        intersection = np.sum(t_mask * p_mask)
        union = np.sum(t_mask) + np.sum(p_mask) - intersection
        iou = 1.0 if (np.sum(t_mask) == 0 and np.sum(p_mask) == 0) else intersection / (union + 1e-6)

        if np.sum(t_mask) == 0 and np.sum(p_mask) == 0: hd = 0.0
        elif np.sum(t_mask) == 0 or np.sum(p_mask) == 0: hd = MAX_HD_PENALTY
        else:
            try: hd = hd95(p_mask, t_mask, voxelspacing=(1,1))
            except: hd = MAX_HD_PENALTY

        # 2. Object Matching (WITH WATERSHED)
        p_labels = separate_touching_cells(p_mask, min_dist=20)
        t_labels = separate_touching_cells(t_mask, min_dist=20) # Apply to truth for fairness

        p_props = [r for r in measure.regionprops(p_labels) if r.area > MIN_AREA]
        t_props = [r for r in measure.regionprops(t_labels) if r.area > MIN_AREA]

        c_true = len(t_props)
        c_pred = len(p_props)
        count_mae = abs(c_true - c_pred)

        p_cents = [r.centroid for r in p_props]
        t_cents = [r.centroid for r in t_props]

        tp_curr = 0
        if t_cents and p_cents:
            dists = cdist(np.array(t_cents), np.array(p_cents))
            r, c = linear_sum_assignment(dists)
            tp_curr = sum(1 for row, col in zip(r, c) if dists[row, col] < MAX_DISTANCE)

        total_tp += tp_curr
        total_fp += len(p_cents) - tp_curr
        total_fn += len(t_cents) - tp_curr

        # Store Data
        results.append({
            'Model': model_name, 'Image_Idx': i, 'Metric': 'Image_Stats',
            'IoU': iou, 'HD95': hd, 'Count_True': c_true, 'Count_Pred': c_pred, 'Count_MAE': count_mae
        })

        for t_reg in t_props:
            mask_iso = (t_labels == t_reg.label)
            overlap = np.sum(mask_iso & p_mask) / t_reg.area
            results.append({
                'Model': model_name, 'Image_Idx': i, 'Metric': 'Size_Recall',
                'Area': t_reg.area, 'Detected': 1 if overlap > 0.5 else 0
            })

    # Global Object Metrics
    prec = total_tp / (total_tp + total_fp + 1e-6)
    rec = total_tp / (total_tp + total_fn + 1e-6)
    f1 = 2 * prec * rec / (prec + rec + 1e-6)
    print(f"  --> Object F1: {f1:.3f} | Prec: {prec:.3f} | Rec: {rec:.3f}")

    del model; gc.collect(); tf.keras.backend.clear_session()

df_res = pd.DataFrame(results)
print("🏁 ARENA COMBAT COMPLETE.")

In [ ]:
# ==============================================================================
# SECTIONS 1, 3, 9: CORE METRICS, STATISTICAL SIGNIFICANCE & CORRELATION
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import r2_score
from scipy.stats import wilcoxon
import pandas as pd

print("📊 Generating Core Metrics & Statistical Report...")

# Prepare data
df_stats = df_res[df_res['Metric'] == 'Image_Stats'].copy()
df_stats['Model Architecture'] = df_stats['Model'].apply(lambda x: x.split('(')[0].strip())
df_stats['Training Method'] = df_stats['Model'].apply(lambda x: x.split('(')[1].replace(')','').strip())

# --- 1. SOTA SUMMARY TABLE ---
table_data = []
for model in df_stats['Model'].unique():
    subset = df_stats[df_stats['Model'] == model]

    iou_mean = subset['IoU'].mean()
    hd95_mean = subset['HD95'].mean()
    mae_mean = subset['Count_MAE'].mean()
    r2 = r2_score(subset['Count_True'], subset['Count_Pred'])

    table_data.append({
        'Model Architecture': model.split('(')[0].strip(),
        'Training Method': model.split('(')[1].replace(')',''),
        'Mean Pixel IoU': f"{iou_mean:.3f}",
        'Boundary Error (HD95) ↓': f"{hd95_mean:.1f} px",
        'Count Error (MAE) ↓': f"{mae_mean:.2f} cells",
        'Count Correlation (R²) ↑': f"{r2:.3f}"
    })

print("\n" + "="*90)
print("🏆 TABLE 1: QUANTITATIVE MODEL COMPARISON")
display(pd.DataFrame(table_data).sort_values(by=['Model Architecture', 'Training Method']))
print("="*90)

# --- 2. STATISTICAL SIGNIFICANCE (Wilcoxon Test) ---
print("\n🔬 STATISTICAL SIGNIFICANCE TESTING (Curriculum vs Baseline):")
for arch in ["TransUNet", "U-Net++", "U-Net"]:
    try:
        curr_ious = df_stats[df_stats['Model'] == f"{arch} (Curriculum)"]['IoU'].values
        base_ious = df_stats[df_stats['Model'] == f"{arch} (Baseline)"]['IoU'].values
        stat, p_val = wilcoxon(curr_ious, base_ious)
        sig = "*** (p < 0.001)" if p_val < 0.001 else "** (p < 0.01)" if p_val < 0.01 else "* (p < 0.05)" if p_val < 0.05 else "Not Sig"
        print(f"   -> {arch}: p-value = {p_val:.4e} {sig}")
    except ValueError: pass

# --- 3. COUNT CORRELATION (R² SCATTER PLOTS) ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, model in enumerate(df_stats['Model'].unique()):
    sub = df_stats[df_stats['Model'] == model]
    r2 = r2_score(sub['Count_True'], sub['Count_Pred'])
    sns.regplot(data=sub, x='Count_True', y='Count_Pred', ax=axes[i], scatter_kws={'alpha':0.4, 'color':'teal'}, line_kws={'color':'red'})

    max_val = max(sub['Count_True'].max(), sub['Count_Pred'].max())
    if np.isnan(max_val) or max_val == 0: max_val = 10
    axes[i].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Perfect Agreement')
    axes[i].set_title(f"{model}\n$R^2$ = {r2:.3f}", fontsize=12)
    axes[i].set_xlabel("Pathologist Count (Truth)"); axes[i].set_ylabel("AI Count (Pred)")
    axes[i].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# SECTIONS 3, 4, 5: DISTRIBUTIONS, BOUNDARIES (HD95), & ERROR MODES
# ==============================================================================
print("📈 Analyzing Distributions and Error Modes...")

df_stats['False_Positives'] = np.maximum(0, df_stats['Count_Pred'] - df_stats['Count_True'])
df_stats['False_Negatives'] = np.maximum(0, df_stats['Count_True'] - df_stats['Count_Pred'])

fig = plt.figure(figsize=(22, 6))

# --- PLOT 1: CDF Curves (Distributional Robustness) ---
ax1 = plt.subplot(1, 3, 1)
sns.ecdfplot(data=df_stats, x='IoU', hue='Model', palette='tab10', ax=ax1, linewidth=2.5)
ax1.set_title("CDF of Image IoU\n(Further Right = More Robust)", fontsize=14)
ax1.set_xlabel("Pixel IoU Score"); ax1.set_ylabel("Proportion of Test Dataset")
ax1.grid(alpha=0.3)

# --- PLOT 2: Boundary Analysis (HD95 Violin Plot) ---
ax2 = plt.subplot(1, 3, 2)
valid_hd = df_stats[df_stats['HD95'] < 400] # Ignore massive outliers for boundary plot
sns.violinplot(data=valid_hd, x='Model Architecture', y='HD95', hue='Training Method',
               split=True, inner="quart", palette="Set2", ax=ax2, cut=0)
ax2.set_title("Boundary Precision (HD95 Distance)\n(Lower is better = Tighter Edges)", fontsize=14)
ax2.set_ylabel("Hausdorff Distance 95% (Pixels)")
ax2.grid(axis='y', alpha=0.3)

# --- PLOT 3: Error Mode Categorization (Hallucinations vs Misses) ---
ax3 = plt.subplot(1, 3, 3)
error_df = df_stats.groupby('Model')[['False_Positives', 'False_Negatives']].sum().reset_index()
error_df.plot(x='Model', y=['False_Positives', 'False_Negatives'], kind='bar',
              color=['#e74c3c', '#3498db'], ax=ax3)
ax3.set_title("Instance Error Breakdown\n(Hallucinations vs. Misses)", fontsize=14)
ax3.set_ylabel("Total Number of Cells Across Vault")
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=45, ha='right')
ax3.legend(["False Positives (Hallucinations)", "False Negatives (Misses)"])
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# SECTION 8: SUBGROUP PERFORMANCE (BIOLOGICAL SIZE BIAS)
# ==============================================================================
print("🔬 Analyzing Detection Sensitivity by Cell Size...")

df_size = df_res[df_res['Metric'] == 'Size_Recall'].copy()
df_size['Size_Category'] = pd.cut(df_size['Area'], bins=[0, 500, 1500, 10000], labels=['Small (Debris/Artifacts)', 'Medium', 'Large (Active Osteoclasts)'])

plt.figure(figsize=(14, 6))
sns.barplot(data=df_size, x='Size_Category', y='Detected', hue='Model', palette='tab10')
plt.title("Subgroup Analysis: Detection Sensitivity by Biologically Relevant Cell Size", fontsize=15, fontweight='bold')
plt.ylabel("Probability of Detection (Recall)")
plt.xlabel("Cell Size Category")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# CELL 8: ALGORITHM VISUALIZATION (Full 7-Step Pipeline)
# ==============================================================================
from skimage import color
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
import tensorflow as tf
from tensorflow.keras.models import Model
import numpy as np
import cv2

print("⚙️ Generating Full Pipeline Visualization...")

# 1. Match the exact image used in Cell 10
valid_imgs = df_stats[df_stats['Count_True'] > 4]['Image_Idx'].values
vis_idx = valid_imgs[0] if len(valid_imgs) > 0 else 10

img_norm, true_mask, img_disp = load_test_image(vault_img[vis_idx], vault_mask[vis_idx])

# 2. Load Model & Predict
m_name = "TransUNet (Curriculum)"
model = load_contender(m_name)
preds = model.predict(np.expand_dims(img_norm, 0), verbose=0)

# Robustly handle TransUNet's [final, aux1, aux2] list structure
if isinstance(preds, list):
    pred_prob = np.squeeze(preds[0]) # TransUNet final output is index 0
else:
    pred_prob = np.squeeze(preds)

if pred_prob.ndim == 3: pred_prob = pred_prob[:,:,0]

# 3. Extract Grad-CAM Saliency Map (ROBUST GRAPH SEARCH)
def get_gradcam(img_tensor, model):
    # Find the true final output tensor dynamically
    if isinstance(model.output, list):
        out_tensor = next((o for o in model.output if 'final' in o.name.lower()), model.output[0])
    else:
        out_tensor = model.output

    # Get all convolutional layers
    valid_layers = [l.name for l in model.layers if isinstance(l, (tf.keras.layers.Conv2D, tf.keras.layers.Conv2DTranspose))]

    target_layer = None
    saved_grads = None
    saved_conv_outputs = None

    # Search backwards for the deepest layer that has a valid, connected gradient
    for layer_name in reversed(valid_layers):
        if 'output' in layer_name.lower() or 'final' in layer_name.lower() or 'aux' in layer_name.lower():
            continue

        grad_model = Model(model.inputs, [model.get_layer(layer_name).output, out_tensor])
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_tensor)
            loss = predictions[:, :, :, 0]

        grads = tape.gradient(loss, conv_outputs)

        if grads is not None:
            target_layer = layer_name
            saved_grads = grads[0]
            saved_conv_outputs = conv_outputs[0]
            break

    if target_layer is None or saved_grads is None:
        print("⚠️ Warning: Could not find connected layer for Grad-CAM. Graph may be disconnected.")
        return np.zeros((512, 512))

    weights = tf.reduce_mean(saved_grads, axis=(0, 1))
    cam = tf.reduce_sum(tf.multiply(weights, saved_conv_outputs), axis=-1)
    cam = tf.maximum(cam, 0)

    # Safely normalize
    cam_max = tf.reduce_max(cam)
    if cam_max > 0:
        cam = cam / cam_max

    return cv2.resize(cam.numpy(), (512, 512))

heatmap = get_gradcam(np.expand_dims(img_norm, 0), model)
del model; gc.collect(); tf.keras.backend.clear_session()

# 4. Post-Processing (Watershed)
p_mask_bin = (pred_prob > 0.5).astype(np.uint8)
p_mask_filled = ndimage.binary_fill_holes(p_mask_bin).astype(np.uint8)

distance = ndimage.distance_transform_edt(p_mask_filled)
local_maxi = peak_local_max(distance, min_distance=20, labels=p_mask_filled)
mask = np.zeros(distance.shape, dtype=bool)
if len(local_maxi) > 0: mask[tuple(local_maxi.T)] = True
markers, _ = ndimage.label(mask)
p_labels = watershed(-distance, markers, mask=p_mask_filled)
p_colored = color.label2rgb(p_labels, bg_label=0)

# 5. Bipartite Matching Data Prep
t_mask_bin = np.squeeze(true_mask).astype(np.uint8)
t_labels = measure.label(t_mask_bin)

p_props = [r for r in measure.regionprops(p_labels) if r.area > MIN_AREA]
t_props = [r for r in measure.regionprops(t_labels) if r.area > MIN_AREA]
p_cents = [r.centroid for r in p_props]
t_cents = [r.centroid for r in t_props]

# ==============================================================================
# PLOTTING THE 7-STEP STORYBOARD
# ==============================================================================
fig, axes = plt.subplots(1, 7, figsize=(35, 5))
fig.suptitle("Complete AI Inference Pipeline: From Raw Tissue to Object Counting", fontsize=18, fontweight='bold', y=1.05)

# Panel 1: Raw Image
axes[0].imshow(img_disp)
axes[0].set_title("1. Raw Microscope Scan")

# Panel 2: Preprocessed Tensor
img_norm_vis = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min() + 1e-6)
axes[1].imshow(img_norm_vis)
axes[1].set_title("2. Preprocessed Tensor\n(Normalized)")

# Panel 3: Saliency Map
axes[2].imshow(img_disp)
axes[2].imshow(heatmap, cmap='jet', alpha=0.5)
axes[2].set_title("3. Saliency Map\n(Grad-CAM Attention)")

# Panel 4: Prob Map
axes[3].imshow(pred_prob, cmap='viridis')
axes[3].set_title("4. Raw Probability Map")

# Panel 5: Threshold
axes[4].imshow(p_mask_filled, cmap='gray')
axes[4].set_title("5. Binary Threshold\n(& Fill Holes)")

# Panel 6: Watershed
axes[5].imshow(p_colored)
axes[5].set_title("6. Watershed Separation\n(Instance Extraction)")

# Panel 7: Bipartite Match
axes[6].imshow(img_disp)
if t_cents: axes[6].plot(np.array(t_cents)[:, 1], np.array(t_cents)[:, 0], 'bo', markersize=8, fillstyle='none', markeredgewidth=2, label='Pathologist (Truth)')
if p_cents: axes[6].plot(np.array(p_cents)[:, 1], np.array(p_cents)[:, 0], 'rx', markersize=8, markeredgewidth=2, label='AI (Pred)')

if t_cents and p_cents:
    dists = cdist(np.array(t_cents), np.array(p_cents))
    r, c = linear_sum_assignment(dists)
    for row, col in zip(r, c):
        if dists[row, col] < MAX_DISTANCE:
            axes[6].plot([p_cents[col][1], t_cents[row][1]], [p_cents[col][0], t_cents[row][0]], 'g-', linewidth=2)

axes[6].set_title("7. Bipartite Matching\n(Final Cell Count)")
axes[6].legend(loc='upper right', fontsize=8)

for ax in axes: ax.axis('off')
plt.tight_layout()

# Save for the poster!
plt.savefig("Pipeline_7_Step_Storyboard.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# ==============================================================================
# CELL 9: VISUAL INSPECTION (THE ARCHETYPE GRID)
# ==============================================================================
import matplotlib.patches as patches

print("🔍 Hunting for Evaluation Archetypes...")

# 1. Filter out empty images for the "Easy Win"
valid_img_stats = df_stats[df_stats['Count_True'] > 0]

# Find the Archetypes
avg_iou_per_img = valid_img_stats.groupby('Image_Idx')['IoU'].mean()
easy_idx = avg_iou_per_img.idxmax() # Guaranteed to have at least 1 cell

tu_base = valid_img_stats[(valid_img_stats['Model'] == 'TransUNet (Baseline)')]
tu_curr = valid_img_stats[(valid_img_stats['Model'] == 'TransUNet (Curriculum)')]
merged = pd.merge(tu_base, tu_curr, on='Image_Idx', suffixes=('_base', '_curr'))
carry_candidates = merged[(merged['IoU_base'] < 0.4) & (merged['IoU_curr'] > 0.6)]
carry_idx = carry_candidates['Image_Idx'].iloc[0] if not carry_candidates.empty else merged['IoU_curr'].idxmax()

wall_idx = avg_iou_per_img.idxmin() # Lowest IoU where cells exist

archetypes = {
    "Archetype A: The 'Easy Win'\n(Ideal Tissue Conditions)": easy_idx,
    "Archetype B: The 'Curriculum Carry'\n(Domain Adaptation fixes Baseline failure)": carry_idx,
    "Archetype C: The 'Aleatoric Wall'\n(Inherent Biological Ambiguity)": wall_idx
}

# 2. Pre-Calculate Predictions (Fixes the Blank Grid Issue)
print("🧠 Running live inference for targeted visualizations...")
archetype_preds = {idx: {} for idx in archetypes.values()}

models_to_plot = [
    "U-Net (Baseline)", "U-Net++ (Baseline)", "TransUNet (Baseline)",
    "U-Net (Curriculum)", "U-Net++ (Curriculum)", "TransUNet (Curriculum)"
]

for m_name in models_to_plot:
    model = load_contender(m_name)
    for idx in archetypes.values():
        img_norm, _, _ = load_test_image(vault_img[idx], vault_mask[idx])
        preds = model.predict(np.expand_dims(img_norm, 0), verbose=0)

        if MODEL_REGISTRY[m_name]['type'] == 'transunet': pred_prob = preds[0][0,:,:,0]
        elif MODEL_REGISTRY[m_name]['type'] == 'unetplusplus': pred_prob = preds[-1][0,:,:,0]
        else: pred_prob = preds[0,:,:,0]

        archetype_preds[idx][m_name] = np.squeeze(pred_prob)
    del model; gc.collect(); tf.keras.backend.clear_session()

def create_rgb_error_map(truth, pred_prob):
    p_mask = ndimage.binary_fill_holes((pred_prob > 0.5)).astype(np.uint8)
    t_mask = truth.astype(np.uint8)
    error_img = np.zeros((512, 512, 3), dtype=np.float32)
    error_img[ (t_mask==1) & (p_mask==1) ] = [0.2, 0.8, 0.2] # True Pos (Green)
    error_img[ (t_mask==0) & (p_mask==1) ] = [0.9, 0.1, 0.1] # False Pos (Red)
    error_img[ (t_mask==1) & (p_mask==0) ] = [0.1, 0.4, 0.9] # False Neg (Blue)
    return error_img

# 3. Generate the Massive Plots
for title, idx in archetypes.items():
    print(f"\nGenerating Grid for: {title.split(chr(10))[0]}")
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    fig.suptitle(title, fontsize=18, fontweight='bold', y=1.02)

    img_norm, true_mask, img_disp = load_test_image(vault_img[idx], vault_mask[idx])

    # Column 0: Inputs
    axes[0, 0].imshow(img_disp); axes[0, 0].set_title("Raw Tissue", fontsize=14); axes[0, 0].axis('off')
    axes[1, 0].imshow(img_disp); axes[1, 0].imshow(true_mask, cmap='gray', alpha=0.6);
    axes[1, 0].set_title("Pathologist Ground Truth", fontsize=14); axes[1, 0].axis('off')

    # Columns 1-3: Models
    for m_i, arch in enumerate(["U-Net", "U-Net++", "TransUNet"]):
        for r_i, regime in enumerate(["Baseline", "Curriculum"]):
            m_name = f"{arch} ({regime})"
            ax = axes[r_i, m_i + 1]

            pred_prob = archetype_preds[idx][m_name]
            iou = df_stats[(df_stats['Model'] == m_name) & (df_stats['Image_Idx'] == idx)].iloc[0]['IoU']

            err_map = create_rgb_error_map(true_mask, pred_prob)
            ax.imshow(img_disp)
            ax.imshow(err_map, alpha=0.55)
            ax.set_title(f"{m_name}\nIoU: {iou:.2f}", fontsize=12)
            ax.axis('off')

            border_color = 'darkred' if r_i == 0 else 'darkgreen'
            rect = patches.Rectangle((0,0), 511, 511, linewidth=4, edgecolor=border_color, facecolor='none')
            ax.add_patch(rect)

    import matplotlib.lines as mlines
    legend_elements = [
        mlines.Line2D([0], [0], marker='s', color='w', label='True Positive (Hit)', markerfacecolor=[0.2, 0.8, 0.2], markersize=15),
        mlines.Line2D([0], [0], marker='s', color='w', label='False Positive (Hallucination)', markerfacecolor=[0.9, 0.1, 0.1], markersize=15),
        mlines.Line2D([0], [0], marker='s', color='w', label='False Negative (Miss)', markerfacecolor=[0.1, 0.4, 0.9], markersize=15)
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=14, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# CELL 10: MECHANISTIC INTERPRETABILITY (3-WAY ARCHITECTURE COMPARISON)
# ==============================================================================
print("🔬 Extracting Deep Feature Maps (3-Way Architecture Comparison)...")

def get_bottleneck_model(model_name):
    """Dynamically finds the deepest layer of any U-Net variant."""
    model = load_contender(model_name)
    if model is None: return None, None

    if MODEL_REGISTRY[model_name]['type'] == 'transunet':
        # TransUNet: Grab the output of the transformer bottleneck
        target_layer = [l.name for l in model.layers if 'conv2d' in l.name][-6]
    else:
        # Universal CNN Deepest Layer Finder
        spatial_dims = []
        for l in model.layers:
            try:
                # FIXED: Safely unwrap Keras list shapes
                shape = l.output_shape
                if isinstance(shape, list): shape = shape[0]

                if isinstance(shape, tuple) and len(shape) == 4 and shape[1] is not None:
                    name = l.name.lower()
                    if 'conv' in name or 'relu' in name or 'concat' in name or 'activation' in name:
                        spatial_dims.append((shape[1], l.name))
            except:
                continue

        if len(spatial_dims) == 0:
            print(f"   -> Warning: Could not detect spatial dimensions for {model_name}. Using fallback.")
            target_layer = model.layers[len(model.layers)//2].name
        else:
            # Find the absolute bottom of the "U"
            min_dim = min([dim for dim, name in spatial_dims])
            # Grab the last layer before upsampling begins
            target_layer = [name for dim, name in spatial_dims if dim == min_dim][-1]

    try:
        feature_model = Model(inputs=model.inputs, outputs=model.get_layer(target_layer).output)
        return feature_model, model
    except Exception as e:
        print(f"   -> Could not build feature extractor for {model_name}: {e}")
        return None, None

# Pick a clean, obvious image with a good number of cells
valid_imgs = df_stats[df_stats['Count_True'] > 4]['Image_Idx'].values
vis_idx = valid_imgs[0] if len(valid_imgs) > 0 else 10

img_norm, true_mask, img_disp = load_test_image(vault_img[vis_idx], vault_mask[vis_idx])

# We use Curriculum for all 3 to isolate the ARCHITECTURE as the only changing variable
compare_archs = ["U-Net (Curriculum)", "U-Net++ (Curriculum)", "TransUNet (Curriculum)"]
fig, axes = plt.subplots(len(compare_archs), 5, figsize=(22, 14))
fig.suptitle("Internal Feature Representation (Deepest Bottleneck Layer)", fontsize=18, fontweight='bold', y=1.02)

for r_i, m_name in enumerate(compare_archs):
    feat_model, full_model = get_bottleneck_model(m_name)
    if feat_model is None: continue

    # Get features
    features = feat_model.predict(np.expand_dims(img_norm, 0), verbose=0)

    # Safely handle models returning lists (Deep Supervision)
    if isinstance(features, list): features = features[-1]

    features = features[0] # Grab the first item in the batch

    # Isolate the 4 channels with the highest variance (most "active" thoughts)
    channel_vars = [np.var(features[:,:,c]) for c in range(features.shape[-1])]
    top_channels = np.argsort(channel_vars)[-4:][::-1]

    # 1. Plot Input
    axes[r_i, 0].imshow(img_disp)
    axes[r_i, 0].set_ylabel(f"{m_name}\nDeep Features", fontsize=15, fontweight='bold')
    if r_i == 0: axes[r_i, 0].set_title("Input Image", fontsize=14)

    # 2. Plot Top 4 Feature Channels
    for c_i, ch in enumerate(top_channels):
        feat_map = features[:, :, ch]

        # Normalize for display
        feat_min, feat_max = feat_map.min(), feat_map.max()
        if feat_max > feat_min:
            feat_map = (feat_map - feat_min) / (feat_max - feat_min)
        else:
            feat_map = np.zeros_like(feat_map) # Handle dead channels safely

        # Resize up to original size for visual comparison
        feat_map_resized = cv2.resize(feat_map, (512, 512), interpolation=cv2.INTER_CUBIC)

        ax = axes[r_i, c_i+1]
        ax.imshow(feat_map_resized, cmap='magma')
        if r_i == 0: ax.set_title(f"High-Variance Activation", fontsize=14)

        # Add channel label cleanly in the corner
        ax.text(0.05, 0.95, f"Ch: {ch}", color='white', fontsize=12,
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(facecolor='black', alpha=0.6, pad=3, edgecolor='none'))

    tf.keras.backend.clear_session(); del feat_model; del full_model; gc.collect()

for ax in axes.flatten():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(); plt.show()
print("💡 What to look for: Watch how the feature maps evolve from top to bottom. U-Net features will look like noisy grids. U-Net++ will look slightly cleaner but still fragmented. TransUNet will look like isolated, organic cellular blobs.")

In [ ]:
# ==============================================================================
# CELL 11: CLINICAL RELIABILITY (BLAND-ALTMAN ANALYSIS)
# ==============================================================================
print("🩺 Generating Bland-Altman Plots (Clinical Reliability Standard)...")

# We will focus on the three Curriculum models
curr_models = [m for m in df_stats['Model'].unique() if 'Curriculum' in m]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Bland-Altman Agreement: AI vs Pathologist\n(Exposes Systematic Over/Under Counting)", fontsize=16, fontweight='bold', y=1.05)

for i, model in enumerate(curr_models):
    sub = df_stats[df_stats['Model'] == model].copy()

    # Bland-Altman Math
    mean_count = (sub['Count_True'] + sub['Count_Pred']) / 2
    diff_count = sub['Count_Pred'] - sub['Count_True'] # Positive = AI overcounted

    md = np.mean(diff_count)
    sd = np.std(diff_count, axis=0)

    ax = axes[i]
    sns.scatterplot(x=mean_count, y=diff_count, ax=ax, alpha=0.6, color='teal', edgecolor='k')

    # Add agreement lines
    ax.axhline(md, color='red', linestyle='-', linewidth=2, label=f'Mean Bias: {md:.2f}')
    ax.axhline(md + 1.96*sd, color='gray', linestyle='--', label=f'+1.96 SD: {md + 1.96*sd:.2f}')
    ax.axhline(md - 1.96*sd, color='gray', linestyle='--', label=f'-1.96 SD: {md - 1.96*sd:.2f}')
    ax.axhline(0, color='black', linewidth=1) # Perfect agreement zero-line

    ax.set_title(f"{model}")
    ax.set_xlabel("Average Number of Cells (AI + Truth) / 2")
    ax.set_ylabel("Difference (AI Pred - Truth)")
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("💡 Interpretation: Points above 0 mean the AI hallucinated. Points below 0 mean the AI missed cells. A widening 'funnel' shape means the model breaks down in highly dense tissue.")

In [ ]:
# ==============================================================================
# CELL 12: ROBUSTNESS VS. TISSUE DENSITY
# ==============================================================================
print("🧬 Analyzing Model Robustness in Dense Tissue...")

# Create Density Bins based on true count
density_bins = [-1, 5, 15, 100]
density_labels = ['Sparse (0-5)', 'Moderate (6-15)', 'Dense (15+)']

df_density = df_stats.copy()
df_density['Tissue_Density'] = pd.cut(df_density['Count_True'], bins=density_bins, labels=density_labels)

# Calculate Object F1 per image (Approximation for visualization)
df_density['TP'] = np.minimum(df_density['Count_True'], df_density['Count_Pred']) # Rough approx
df_density['Approx_Obj_F1'] = 2 * df_density['TP'] / (df_density['Count_True'] + df_density['Count_Pred'] + 1e-6)

plt.figure(figsize=(14, 6))
sns.boxplot(data=df_density, x='Tissue_Density', y='IoU', hue='Model Architecture', palette='Set2')
plt.title("Performance Degradation in High-Density Tissue", fontsize=15, fontweight='bold')
plt.ylabel("Image-Level Pixel IoU")
plt.xlabel("Biological Context (True Cell Count)")
plt.legend(title="Architecture", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print("💡 Interpretation: Notice if older architectures (U-Net) drop in IoU significantly during 'Dense' scenes due to merging touching cells, whereas advanced models maintain performance.")

In [ ]:
# ==============================================================================
# CELL 13: EXPORT FULL MODEL GRAPHS FOR NETRON VISUALIZATION
# ==============================================================================
import os
import gc
import tensorflow as tf

print("🕸️ Generating Full Model Graphs for Netron.app...")

# We are targeting the 3 Curriculum models for the poster comparison
models_to_export = [
    "U-Net (Curriculum)",
    "U-Net++ (Curriculum)",
    "TransUNet (Curriculum)"
]

export_paths = []

for m_name in models_to_export:
    print(f"\n   -> Processing {m_name}...")

    # 1. Load the full architecture + weights using our SOTA Arena function
    model = load_contender(m_name)

    if model is not None:
        # Clean up the string to make a safe filename
        safe_name = m_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "")

        # 2. Save as .keras (This forces Keras to save architecture + weights)
        export_path = os.path.join(PROJECT_BASE_PATH, f"{safe_name}_Full_Graph.keras")

        # Save the complete model structure
        model.save(export_path)
        export_paths.append(export_path)

        print(f"      ✅ Saved full graph to: {export_path}")

        # Free memory between exports
        del model
        tf.keras.backend.clear_session()
        gc.collect()

print("\n" + "="*70)
print("🎉 EXPORT COMPLETE. NEXT STEPS FOR YOUR POSTER:")
print("="*70)
print("1. Go to your Google Drive (CHANA_files folder) and download:")
for p in export_paths:
    print(f"   - {os.path.basename(p)}")
print("2. Open your web browser and go to: https://netron.app/")
print("3. Drag and drop a downloaded .keras file onto the webpage.")
print("4. Click the menu (top left) -> 'Export as SVG'.")
print("5. Drop the SVG into Draw.io or Figma to trace your architecture diagrams!")

In [ ]:
# ==============================================================================
# CELL 14: THE GIANT CELL BIAS TEST (Size-Stratified IoU)
# ==============================================================================
print("🧪 Running Giant Cell Bias Test...")

# Define Biological Size Bins
# Giant cells (>1500px) are the most important for resorption studies
bins = [0, 500, 1500, 100000]
labels = ['Small/Debris', 'Mature', 'Giant (Highest Activity)']

# Re-using the object_data generated in Cell 11 if available,
# or calculating fresh from the results dataframe
df_size_stats = df_res[df_res['Metric'] == 'Size_Recall'].copy()
df_size_stats['Size_Category'] = pd.cut(df_size_stats['Area'], bins=bins, labels=labels)

# Calculate Mean IoU per Size Category per Model
size_iou_report = df_size_stats.groupby(['Model', 'Size_Category'])['Detected'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=size_iou_report, x='Size_Category', y='Detected', hue='Model', marker='o', linewidth=3)
plt.title("Pharmacological Sensitivity: Detection Reliability vs. Cell Maturity", fontsize=14, fontweight='bold')
plt.ylabel("Recall (Detection Probability)")
plt.xlabel("Biological Maturity Stage")
plt.ylim(0, 1.1)
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("💡 Analysis: High performance in the 'Giant' category proves the model is suitable for bone resorption assays.")

In [ ]:
# ==============================================================================
# CELL 15: TOPOLOGICAL CONSISTENCY (Betti Number Analysis)
# ==============================================================================
print("🧮 Calculating Topological Error (Connected Components)...")

topo_results = []

# Analyze connectivity for Curriculum models
for m_name in ["U-Net (Curriculum)", "U-Net++ (Curriculum)", "TransUNet (Curriculum)"]:
    sub = df_res[(df_res['Model'] == m_name) & (df_res['Metric'] == 'Image_Stats')]

    # Mathematical Connectivity Ratio: Predicted Objects / True Objects
    # Ideal ratio = 1.0. >1.0 means fragmentation (cell split), <1.0 means merging.
    sub['Connectivity_Ratio'] = sub['Count_Pred'] / (sub['Count_True'] + 1e-6)

    # Average Fragmentation Index (Percentage of cells broken into multiple pieces)
    frag_index = sub[sub['Count_Pred'] > sub['Count_True']].shape[0] / len(sub)

    topo_results.append({
        'Model': m_name,
        'Fragmentation_Index': frag_index,
        'Connectivity_Avg': sub['Connectivity_Ratio'].mean()
    })

df_topo = pd.DataFrame(topo_results)
plt.figure(figsize=(10, 5))
sns.barplot(data=df_topo, x='Model', y='Fragmentation_Index', palette='magma')
plt.title("Topological Robustness: Fragmentation Frequency", fontsize=14, fontweight='bold')
plt.ylabel("Index (Lower = More Biologically Coherent)")
plt.show()

print("💡 AI insight: U-Net++'s dense skip pathways should show the lowest fragmentation compared to TransUNet.")

In [ ]:
# ==============================================================================
# CELL 16: ENHANCED BLAND-ALTMAN (Tissue Density Bias Test)
# ==============================================================================
print("🩺 Generating High-Density Reliability Plot...")

m_target = "U-Net++ (Curriculum)"
sub = df_stats[df_stats['Model'] == m_target].copy()

# Math
mean_val = (sub['Count_True'] + sub['Count_Pred']) / 2
diff_val = sub['Count_Pred'] - sub['Count_True']
md = np.mean(diff_val)
sd = np.std(diff_val)

plt.figure(figsize=(12, 7))
# Color by true count to show density effects
sc = plt.scatter(mean_val, diff_val, c=sub['Count_True'], cmap='viridis', alpha=0.7, edgecolors='w', s=80)
plt.colorbar(sc, label='True Cell Count (Density)')

# 95% Confidence Intervals
plt.axhline(md, color='red', linestyle='-', linewidth=2, label=f'Mean Bias: {md:.2f}')
plt.axhline(md + 1.96*sd, color='gray', linestyle='--', alpha=0.8, label='Upper 95% LoA')
plt.axhline(md - 1.96*sd, color='gray', linestyle='--', alpha=0.8, label='Lower 95% LoA')

plt.title(f"Clinical Agreement: {m_target}\nColor-coded by Tissue Confluency", fontsize=15, fontweight='bold')
plt.xlabel("Average Count (Pathologist + AI) / 2")
plt.ylabel("Error (AI - Pathologist)")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.2)
plt.show()